# GSE268904: extract, inventory and QC (R, Google Colab)

Human subcutaneous adipose snRNA-seq (10x), 14 samples: lean (4), obese (7), obese diabetic (3).
This notebook extracts `GSE268904_RAW.tar`, builds a checksummed file inventory, reads each sample's 10x matrix, reports nuclei counts and QC metrics, and makes a **crude** endothelial estimate. An optional last section clusters and annotates for a better endothelial count.

**Before running**
1. Runtime > Change runtime type > **R**.
2. Open the folder icon on the left, click the Drive icon to **mount Google Drive**.
3. Run the cells in order. Only the small output files (CSV, markdown) need to be shared; the raw data stay in Colab.

*Status: written without being able to run R, so expect small fixes on a first run. Paste any error message back to the assistant.*

## 1. Configuration

In [ ]:
# Edit TAR_PATH after you find the file in the next cell
TAR_PATH <- ""                                   # e.g. "/content/drive/MyDrive/GSE268904_RAW.tar"
WORK_DIR <- "/content/gse268904_raw"             # extracted files (treated as read-only)
OUT_DIR  <- "/content/gse268904_qc"              # small outputs to download or copy to Drive
MIN_UMI  <- 500                                  # barcodes below this total UMI are not counted as nuclei
ENDO_MARKERS <- c("PECAM1", "VWF", "CLDN5", "EGFL7", "FLT1", "PTPRB")  # crude endothelial panel
RUN_SEURAT <- FALSE                              # TRUE = also run the optional clustering section (slow)
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)

## 2. Find the tar in Drive

In [ ]:
hits <- list.files("/content/drive", pattern = "GSE268904", recursive = TRUE, full.names = TRUE)
print(hits)
# Pick the .tar path and set TAR_PATH in the previous cell, or do it here:
if (TAR_PATH == "") { tars <- hits[grepl("\\.tar$", hits)]; if (length(tars) >= 1) TAR_PATH <- tars[1] }
cat("Using:", TAR_PATH, "\n")
stopifnot(file.exists(TAR_PATH))
cat("Size (GB):", round(file.info(TAR_PATH)$size / 1e9, 3), "\n")

## 3. Install packages

In [ ]:
need <- c("Matrix", "data.table", "digest")
for (p in need) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
suppressPackageStartupMessages({library(Matrix); library(data.table); library(digest)})

## 4. Extract the archive and build the file inventory (SHA-256)

In [ ]:
untar(TAR_PATH, exdir = WORK_DIR)
files <- list.files(WORK_DIR, recursive = TRUE, full.names = TRUE)
files <- files[!file.info(files)$isdir]
inv <- data.table(
  relative_path = sub(paste0("^", WORK_DIR, "/?"), "", files),
  filename = basename(files),
  size_bytes = file.info(files)$size,
  sha256 = vapply(files, function(f) digest(f, algo = "sha256", file = TRUE), character(1)),
  file_type = ifelse(grepl("barcodes", files), "barcodes",
              ifelse(grepl("features|genes", files), "features",
              ifelse(grepl("matrix", files), "matrix", "other")))
)
inv[, gsm := sub("^(GSM[0-9]+).*", "\\1", filename)]
inv[, duplicate_of_n := .N, by = sha256]
fwrite(inv, file.path(OUT_DIR, "file_inventory.csv"))
cat("Files extracted:", nrow(inv), " (expected 42 = 14 samples x 3 files)\n")
print(inv[, .N, by = file_type])
cat("Duplicate-content files:", sum(inv$duplicate_of_n > 1), "\n")

## 5. Sample metadata (copied from the GEO sample records)

In [ ]:
meta <- data.table(
  gsm = sprintf("GSM%d", 8302405:8302418),
  sample = c("9S","10S","13S","17S","SAT22","SAT54","SAT1","SAT18","SAT34","SAT39","SAT4","SAT51","SAT58","SAT60"),
  condition = c("obese","obese diabetic","lean","obese diabetic","obese diabetic","lean","obese","obese","obese","obese","obese","obese","lean","lean"),
  sex = c("female","female","female","female","male","female","male","female","male","male","male","female","male","male"),
  age = c(37,41,42,34,50,61,43,29,40,41,30,44,54,26),
  bmi = c(39.5,61.5,23,52.6,39.7,22.3,41.7,61.3,62.5,61.2,49.7,57,24.3,23.5))
fwrite(meta, file.path(OUT_DIR, "samples_metadata.csv"))
# Referential check: every metadata GSM must have 3 files
chk <- inv[, .(n_files = .N, has_barcodes = any(file_type == "barcodes"), has_features = any(file_type == "features"), has_matrix = any(file_type == "matrix")), by = gsm]
print(merge(meta[, .(gsm, sample)], chk, by = "gsm", all = TRUE))

## 6. Per-sample nuclei counts, QC metrics and crude endothelial estimate

In [ ]:
mito_prefix <- "^MT-"
res <- list()
for (i in seq_len(nrow(meta))) {
  gsm <- meta$gsm[i]; smp <- meta$sample[i]
  pick <- function(type) files[grepl(gsm, files) & grepl(type, files)]
  fb <- pick("barcodes"); ff <- pick("features|genes"); fm <- pick("matrix")
  if (length(fb) != 1 || length(ff) != 1 || length(fm) != 1) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "missing files"); next }
  m <- tryCatch(readMM(fm), error = function(e) NULL)
  if (is.null(m)) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "matrix unreadable"); next }
  bc <- fread(fb, header = FALSE)[[1]]; feat <- fread(ff, header = FALSE); gene <- feat[[2]]
  if (ncol(m) != length(bc) || nrow(m) != length(gene)) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "dimension mismatch"); next }
  m <- as(m, "CsparseMatrix")
  umi <- Matrix::colSums(m); ngene <- Matrix::colSums(m > 0)
  mt <- Matrix::colSums(m[grepl(mito_prefix, gene), , drop = FALSE]) / pmax(umi, 1)
  present <- ENDO_MARKERS[ENDO_MARKERS %in% gene]
  det <- if (length(present)) Matrix::colSums(m[match(present, gene), , drop = FALSE] > 0) else rep(0, ncol(m))
  keep <- umi >= MIN_UMI
  res[[i]] <- data.table(gsm = gsm, sample = smp, condition = meta$condition[i], status = "ok",
    n_barcodes_file = length(bc), n_genes_file = length(gene), n_nuclei_umi_ge_min = sum(keep),
    median_umi = median(umi[keep]), median_genes = median(ngene[keep]), median_mito_frac = round(median(mt[keep]), 4),
    raw_counts_integer = all(m@x == round(m@x)),
    markers_present = paste(present, collapse = "|"),
    endo_ge2 = sum(keep & det >= 2), endo_ge3 = sum(keep & det >= 3), endo_ge4 = sum(keep & det >= 4))
  cat(smp, ":", length(bc), "barcodes;", sum(keep), "with UMI >=", MIN_UMI, "\n")
  rm(m); gc(verbose = FALSE)
}
per_sample <- rbindlist(res, fill = TRUE)
fwrite(per_sample, file.path(OUT_DIR, "per_sample_counts.csv"))
print(per_sample)

## 7. Group totals

In [ ]:
ok <- per_sample[status == "ok"]
group <- ok[, .(n_samples = .N, nuclei = sum(n_nuclei_umi_ge_min), endo_ge2 = sum(endo_ge2), endo_ge3 = sum(endo_ge3), endo_ge4 = sum(endo_ge4)), by = condition]
overall <- data.table(condition = "ALL", n_samples = nrow(ok), nuclei = sum(ok$n_nuclei_umi_ge_min), endo_ge2 = sum(ok$endo_ge2), endo_ge3 = sum(ok$endo_ge3), endo_ge4 = sum(ok$endo_ge4))
group <- rbind(group, overall)
fwrite(group, file.path(OUT_DIR, "group_summary.csv"))
print(group)
cat("\nCRUDE estimate: nuclei where >= k of the endothelial markers are detected. Not a cell-type annotation.\n")

## 8. (Optional) Seurat clustering for a better endothelial count
Set `RUN_SEURAT <- TRUE` in section 1 and re-run from here. This installs Seurat (can take 10+ minutes) and clusters each sample separately. Endothelial clusters are called when the cluster's median module score is high and PECAM1/VWF/CLDN5 are detected in a large fraction of its nuclei. **Check the output by eye** before trusting the counts. Per-sample clustering is a shortcut; a proper analysis integrates samples with batch in the model.

In [ ]:
if (RUN_SEURAT) {
  if (!requireNamespace("Seurat", quietly = TRUE)) install.packages("Seurat", repos = "https://cloud.r-project.org")
  suppressPackageStartupMessages(library(Seurat))
  seu_res <- list()
  for (i in seq_len(nrow(meta))) {
    gsm <- meta$gsm[i]; smp <- meta$sample[i]
    pick <- function(type) files[grepl(gsm, files) & grepl(type, files)]
    m <- readMM(pick("matrix")); bc <- fread(pick("barcodes"), header = FALSE)[[1]]; gene <- fread(pick("features|genes"), header = FALSE)[[2]]
    rownames(m) <- make.unique(gene); colnames(m) <- bc
    obj <- CreateSeuratObject(m, min.cells = 3, min.features = 200)
    obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
    obj <- subset(obj, subset = nCount_RNA >= MIN_UMI & percent.mt < 10)
    obj <- NormalizeData(obj, verbose = FALSE); obj <- FindVariableFeatures(obj, nfeatures = 2000, verbose = FALSE)
    obj <- ScaleData(obj, verbose = FALSE); obj <- RunPCA(obj, npcs = 30, verbose = FALSE)
    obj <- FindNeighbors(obj, dims = 1:20, verbose = FALSE); obj <- FindClusters(obj, resolution = 0.5, verbose = FALSE)
    genes_ok <- intersect(ENDO_MARKERS, rownames(obj))
    obj <- AddModuleScore(obj, features = list(genes_ok), name = "endo", verbose = FALSE)
    key <- intersect(c("PECAM1", "VWF", "CLDN5"), rownames(obj))
    pct <- sapply(levels(obj$seurat_clusters), function(cl) mean(colSums(GetAssayData(obj, layer = "counts")[key, obj$seurat_clusters == cl, drop = FALSE] > 0) >= 1))
    sc  <- tapply(obj$endo1, obj$seurat_clusters, median)
    endo_clusters <- names(sc)[sc > 0.3 & pct[names(sc)] > 0.3]
    n_endo <- sum(obj$seurat_clusters %in% endo_clusters)
    seu_res[[i]] <- data.table(sample = smp, condition = meta$condition[i], nuclei_after_qc = ncol(obj), n_clusters = length(levels(obj$seurat_clusters)),
                               endo_clusters = paste(endo_clusters, collapse = "|"), endothelial_nuclei = n_endo)
    cat(smp, ": ", ncol(obj), "nuclei; endothelial clusters:", paste(endo_clusters, collapse = ","), "->", n_endo, "\n")
    rm(obj, m); gc(verbose = FALSE)
  }
  seu <- rbindlist(seu_res); fwrite(seu, file.path(OUT_DIR, "seurat_endothelial_counts.csv")); print(seu)
  print(seu[, .(nuclei = sum(nuclei_after_qc), endothelial = sum(endothelial_nuclei)), by = condition])
} else cat("Skipped (RUN_SEURAT = FALSE)\n")

## 9. Notes file and save outputs to Drive

In [ ]:
writeLines(c("# Notes", "",
  "- Nuclei = barcodes with total UMI >= MIN_UMI. n_barcodes_file is the number of barcodes in the deposited file.",
  "- endo_ge2/3/4 = nuclei where at least k of six endothelial genes are detected: a crude estimate, not an annotation.",
  "- Ambient RNA can inflate endo_ge2; treat ge3 and ge4 as more conservative.",
  "- Optional Seurat counts use per-sample clustering and a simple cluster rule; inspect before use.",
  "- No rows, samples or genes were removed from the raw files.", "", capture.output(sessionInfo())),
  file.path(OUT_DIR, "notes.md"))
cat("Output files:\n"); print(list.files(OUT_DIR))
cat("Total size (MB):", round(sum(file.info(list.files(OUT_DIR, full.names = TRUE))$size) / 1e6, 3), "\n")
# Copy to Drive so you can download from Drive
dest <- "/content/drive/MyDrive/gse268904_qc"
dir.create(dest, showWarnings = FALSE); file.copy(list.files(OUT_DIR, full.names = TRUE), dest, overwrite = TRUE)
cat("Copied to", dest, "- upload these files back to the chat\n")